# 05 · Final RAG: Qdrant text + ColPali (mean pooling) + agent, as in the design

This follows `docs/financial_rag_final_design.md`, sections 3–5, through `scripts/rag/financial_ingest.py`:

| Collection · vector | Content | Index |
|---|---|---|
| `annual_reports_text` · `dense` + `bm25` | every readable page: text chunks and table chunks (OCR for garbled pages) | HNSW + sparse IDF |
| `annual_reports_visual` · `mean_rows`, `mean_cols` | 38 × 128 pooled ColPali vectors per visual page | HNSW (first stage) |
| `annual_reports_visual` · `colpali` | full 1030 × 128 vectors | no graph; exact MaxSim rerank |
| `annual_reports_visual` · `summary` | text vector of the page | HNSW |

**Tested:**
1. Ingest a few filings: text, OCR and visual.
2. Visual retrieval in three modes: **pooled-only**, **pooled prefetch → full rerank** (the design's choice) and **full exhaustive**. Compared on rank and speed.
3. Hybrid text + visual evidence.
4. The **LangGraph financial agent** end to end, after the **clarifier**, scored on the eval set, with grounding.

**Runtime:** **GPU** recommended (ColPali and OCR). Without one, a fake encoder keeps the pipeline runnable.

**Scope:** for the full corpus, Drive snapshots and FastAPI + Streamlit serving, use `notebooks/colab_financial_ingest.ipynb`.

In [ ]:
# 0. Setup. On Colab: unpacks the code zip from Drive (else clones the repo) and installs packages. Locally: run from inside the repo.
import json, os, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/rama159065/Capstone.git"
BRANCH = "finance_report_qa_agent"
CODE_ZIP = "/content/drive/MyDrive/capstone_rag/capstone_code.zip"   # python -m scripts.common.colab_bundle, then upload to Drive
IN_COLAB = "google.colab" in sys.modules


def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


def fetch_code(root, repo_url, branch):
    """Unpack the code zip (Drive, else one you upload) into root; without a zip, clone the repo."""
    import zipfile
    from google.colab import drive
    drive.mount("/content/drive")
    uploaded = Path("/content") / Path(CODE_ZIP).name
    source = next((p for p in (Path(CODE_ZIP), uploaded) if p.exists()), None)
    if source is None and not root.exists():
        from google.colab import files
        print(f"{CODE_ZIP} not found: upload capstone_code.zip (cancel to clone {repo_url} instead)")
        names = list(files.upload())
        source = Path("/content") / names[0] if names else None
    if source is not None:
        with zipfile.ZipFile(source) as archive:
            info = archive.read("BUNDLE_INFO.json")
            stamp = root / "BUNDLE_INFO.json"
            if not stamp.exists() or stamp.read_bytes() != info:      # a new build: unpack over the old one
                archive.extractall(root)
        print("code:", source, json.loads(info))
    elif not root.exists():
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")      # needed while the repo is private
        except Exception:
            token = None
        url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url, str(root)], check=True)


if IN_COLAB:
    ROOT = Path("/content/Capstone")
    fetch_code(ROOT, REPO_URL, BRANCH)
    import re, tomllib                                  # the agent needs the project's full dependency list
    deps = tomllib.load(open(ROOT / "pyproject.toml", "rb"))["project"]["dependencies"]
    deps = [d for d in deps if re.split(r"[<>=!~ \[]", d)[0] not in {"torch", "torchvision", "jupyterlab"}]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)
    for key in ("LLM_API_BASE", "LLM_API_KEY", "LLM_MODEL", "SEC_USER_AGENT"):
        if not os.getenv(key) and colab_secret(key):
            os.environ[key] = colab_secret(key)
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("SEC_USER_AGENT", "capstone-research your.name@example.com")   # SEC requires your real e-mail
print("repo:", ROOT, "| Colab:", IN_COLAB)

**LLM.** `NB_LLM=auto` picks, in order:
1. **The provided Track B endpoint**, when `LLM_API_BASE` and `LLM_MODEL` are set (Colab secrets or environment). Grade with this one.
2. **A small Hugging Face stand-in** (`Qwen/Qwen2.5-1.5B-Instruct`), on a Colab GPU.
3. **The deterministic mock** otherwise. It only proves that the pipeline runs.

Force a choice with `os.environ["NB_LLM"] = "hosted" | "transformers" | "mock"` before the config cell.

The agent itself supports `hosted` (the provided endpoint) or `mock`.

In [ ]:
import numpy as np
from scripts.common import notebook_helpers as nb
from scripts.rag import financial_ingest as fi

DOCS = ["AMD_2025", "ABBV_2025", "NVDA_2026", "CVX_2022"]
WORK = Path(os.getenv("NB_WORK", "/content/nb05_final" if IN_COLAB else "tmp/nb05_final"))
MODE = os.getenv("NB_COLPALI", "auto")
MODE = ("real" if nb._gpu() else "fake") if MODE == "auto" else MODE
EMBEDDING = os.getenv("NB_EMBED", "local")
EMBED_MODEL = os.getenv("NB_EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
OCR_LIMIT = os.getenv("NB_OCR_LIMIT", "0" if MODE == "real" else "20")   # 0 = every garbled page
MAX_PAGES = int(os.getenv("NB_MAX_PAGES", "0")) or None
encoder = fi.ColPali(os.getenv("NB_COLPALI_MODEL", "vidore/colpali-v1.3-hf")) if MODE == "real" else nb.HashColPali()
rows = nb.pick_filings(DOCS)
pdfs = nb.fetch_pdfs(rows)
print("ColPali:", MODE, "| OCR limit per filing:", OCR_LIMIT)

## 1. Ingest: text + OCR + visual (pooled and full vectors)

In [ ]:
WORK.mkdir(parents=True, exist_ok=True)
manifest = WORK / "manifest.jsonl"
manifest.write_text("".join(json.dumps(r) + "\n" for r in rows), encoding="utf-8")
argv = ["ingest", "--root", str(WORK), "--manifest", str(manifest), "--pdf-dir", str(nb.PDF_DIR),
        "--stages", "text,visual", "--ocr", os.getenv("NB_OCR", "garbled"), "--ocr-max-pages-per-filing", OCR_LIMIT,
        "--embedding-provider", EMBEDDING, "--embedding-model", EMBED_MODEL, "--workers", "1", "--reset"]
argv += ["--max-pages-per-pdf", str(MAX_PAGES)] if MAX_PAGES else []
started = time.time()
ingest = fi.Ingest(fi.build_parser().parse_args(argv), colpali=encoder)
print(ingest.run(), f"in {time.time() - started:.0f}s")
client, embedder = ingest.client, ingest.embedder
nb.show(nb.table([{"filing": r["document_id"],
                   "text_points": client.count(fi.TEXT, count_filter=fi.doc_filter(r["document_id"]), exact=True).count,
                   "visual_pages": client.count(fi.VISUAL, count_filter=fi.doc_filter(r["document_id"]), exact=True).count}
                  for r in rows]))

## 2. Visual retrieval: pooled-only vs pooled → full rerank vs full exhaustive

In [ ]:
def visual(mode, query, k=10):
    mv = np.asarray(encoder.embed_query(query), np.float32).tolist()
    started = time.perf_counter()
    if mode == "pooled_rows_only":
        points = client.query_points(fi.VISUAL, query=mv, using="mean_rows", limit=k).points
    elif mode == "pooled_prefetch_full_rerank":
        points = fi.search_visual(client, encoder, embedder, query, k)
    else:
        points = client.query_points(fi.VISUAL, query=mv, using="colpali", limit=k).points
    return [(p.payload["doc_id"], p.payload["page"]) for p in points], time.perf_counter() - started


chart_items = nb.eval_items(DOCS, categories=("chart_value",), general=False)
queries = [(i["question"], (s["doc_id"], s["page"])) for i in chart_items for s in i["sources"] if s.get("page")]
queries += [("bar chart comparing revenue with peer companies", ("edgar_ars_NVDA_2026", 60)),
            ("oil and gas production by region chart", ("edgar_ars_CVX_2022", 35))]
rows_out = []
for query, target in queries:
    full, _ = visual("full_exhaustive", query, 10)
    for mode in ("pooled_rows_only", "pooled_prefetch_full_rerank", "full_exhaustive"):
        ranked, seconds = visual(mode, query, 10)
        rows_out.append({"mode": mode, "query": query[:60], "rank": ranked.index(target) + 1 if target in ranked else None,
                         "overlap@5_with_full": len(set(ranked[:5]) & set(full[:5])) / 5, "ms": round(seconds * 1000, 1)})
cmp = nb.table(rows_out)
nb.show(cmp.groupby("mode").agg(hit_at_1=("rank", lambda r: (r == 1).mean()), hit_at_5=("rank", lambda r: r.le(5).mean()),
                                overlap5_with_full=("overlap@5_with_full", "mean"), median_ms=("ms", "median")).round(2))

## 3. Hybrid evidence for the agent: text + visual pages, fused

In [ ]:
os.environ.update(RAG_BACKEND="financial_qdrant", RAG_INGEST_DIR=str(WORK), RAG_FIN_VISUAL="1")
fi._serving.clear()
fi._serving.update(client=client, embedder=embedder, colpali=encoder)
for query, ticker, year in [("NVIDIA revenue compared with peer companies chart", "NVDA", 2026),
                            ("AMD total net revenue 2025", "AMD", 2025)]:
    hits = fi.retrieve_for_agent(query, top_k=5, ticker=ticker, year=year)
    print(query, "->", [(h["page"], h["metadata"].get("content_type"), "+".join(h["retrieval_paths"])) for h in hits])

## 4. Clarifier, then the LangGraph agent, end to end

In [ ]:
from scripts.agents import financial_prompting_agent as agent
from scripts.agents.financial_clarifier import Clarifier, ConversationStore

AGENT_LLM = "hosted" if os.getenv("LLM_API_BASE") and os.getenv("LLM_MODEL") else "mock"
clarifier, store = Clarifier(), ConversationStore()


def ask(query, ticker=None, year=None):
    config = agent.AgentConfig(query=query, ticker=ticker, year=year, source_type=None, top_k=5, dense_top_k=30,
                               bm25_top_k=30, retrieval_mode="hybrid", embedding_provider="hashing",
                               hash_dimensions=384, max_context_chars=12000, self_consistency_samples=1)
    return agent.run_technique("agentic_graph", AGENT_LLM, config)


conversation = store.get()
for turn in ["What was AMD's revenue?", "FY2025"]:
    outcome = clarifier.step(conversation, turn)
    print(f"user: {turn!r} -> {outcome['status']}: {outcome.get('message') or outcome.get('resolved_query')}")
result = ask(outcome["resolved_query"], outcome["ticker"], outcome["year"])
print("agent:", result.get("final_status"), "| citations valid:", result.get("citation_check", {}).get("valid"),
      "\n", (result.get("answer") or "")[:400])

In [ ]:
items = nb.eval_items(DOCS)[: int(os.getenv("NB_QUESTIONS", "0")) or None]
rows_out = []
for item in items:
    outcome = clarifier.step(store.get(), item["question"])
    # Eval questions are self-contained, so when the clarifier would ask, run the agent on the question as asked
    # and record what the clarifier would have said.
    asked = outcome["status"] == "needs_clarification"
    query = item["question"] if asked else outcome.get("resolved_query") or item["question"]
    result = ask(query, None if asked else outcome.get("ticker"), None if asked else outcome.get("year"))
    hits = nb.label_hits(result.get("evidence") or [])
    scored = nb.score(item, {"text": result.get("answer") or "", "parsed": None, "strategy": "zero_shot"}, hits)
    rows_out.append({"id": item["id"], "category": item["category"], "clarifier": ("asked: " + outcome["message"][:50]) if asked else outcome["status"],
                     "agent_status": result.get("final_status"), "correct": scored["correct"], "failure": scored["failure"],
                     "gold_value_in_evidence": scored["gold_value_in_evidence"],
                     "citations_valid": (result.get("citation_check") or {}).get("valid")})
final = nb.table(rows_out)
nb.show(final.groupby("category")[["correct"]].mean().round(2).join(final.groupby("category").size().rename("n")))
nb.show(final.groupby(["agent_status", "citations_valid"], dropna=False).size().rename("answers"))
nb.show(final[final.clarifier.str.startswith("asked")][["id", "category", "clarifier", "correct"]])
nb.show(final[~final.correct].groupby(["category", "failure"]).size().rename("failures"))

## Failure scenarios for the final pipeline

| Layer | Failure | Mitigation in the design |
|---|---|---|
| Clarifier | A question without a year or company returns `needs_clarification`, so the agent doesn't run (by design). A wrong default assumption, e.g. "latest year", gives the wrong period | The UI asks the follow-up question; assumptions are shown with the answer |
| Visual first stage | Pooled rows/columns miss a page that full MaxSim would rank first (`hit_at_5` of pooled-only vs prefetch → rerank) | Prefetch 200 candidates from rows, columns and summary, then rerank on full vectors |
| Text retrieval | Garbled pages without OCR; statement figures beyond the OCR page limit; wrong-company evidence without filters | OCR every unreadable page; filter by ticker and year; take figures from the XBRL facts tool |
| Fusion | Visual pages carry little text, so the LLM receives an image page it cannot read | Chart values via OCR or validated reading (T3/T4), or say "not printed" |
| Agent | The planner or critic loop with a small model: invented numbers, missing citations, over-refusal | Citation critic → repair loop; grounding policy withholds unsupported answers |
| Grounding | Citations validate against retrieved IDs (consistency), not truth | Number grounding check (design layer 6, not built yet) |
| Mock LLM | Without `LLM_API_BASE` the agent runs on the mock: answers are placeholders | Configure the provided endpoint for real scores |
| Agent classifier fallback | When the classifier returns no usable JSON (always with the mock, sometimes with a small model), the keyword fallback lacks terms like EPS, equity, margin, auditor, production and gross profit. Legitimate questions are then refused (15 of 32 refused in a mock run, only 8 of them true refusals) | Widen `FINANCIAL_INTENT_TERMS`, or use the clarifier's slot parser as the fallback |
| Input guard (fixed) | "Ignore **your** previous instructions…" (FIN-082) passed the injection regex; it was caught only because the classifier refused it | Pattern widened in `scripts/guardrails/patterns.py`; test added |